In [9]:
import pandas as pd
import duckdb
con = duckdb.connect(database=':memory:')

In [3]:
url_trip = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet'
url_taxi = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv'

In [4]:
df_trip = pd.read_parquet(url_trip)
df_taxi = pd.read_csv(url_taxi)

In [5]:
df_taxi.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [6]:
df_trip.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [13]:
q3 = df_trip[
    (df_trip["lpep_pickup_datetime"] >= "2025-11-01") &
    (df_trip["lpep_pickup_datetime"] < "2025-12-01") &
    (df_trip["trip_distance"] <= 1)
].shape[0]

print("Q3: Short trips <= 1 mile:", q3)

Q3: Short trips <= 1 mile: 8007


In [14]:
# Exclude trips >= 100 miles
df_filtered = df_trip[df_trip["trip_distance"] < 100]

# Extract pickup date
df_filtered["pickup_date"] = df_filtered["lpep_pickup_datetime"].dt.date

# Find row with max trip_distance
idx_max = df_filtered["trip_distance"].idxmax()
longest_trip_day = df_filtered.loc[idx_max, "pickup_date"]

print("Q4: Pickup day with longest trip:", longest_trip_day)


Q4: Pickup day with longest trip: 2025-11-14


In [17]:
df_nov18 = df_trip[
    (df_trip["lpep_pickup_datetime"] >= "2025-11-18") &
    (df_trip["lpep_pickup_datetime"] < "2025-11-19")
]

# Sum total_amount by pickup location
pickup_total = df_nov18.groupby("PULocationID")["total_amount"].sum()

# Find the pickup zone with largest total_amount
max_zone_id = pickup_total.idxmax()
zones_dict = dict(zip(df_taxi["LocationID"], df_taxi["Zone"]))
max_zone_name = zones_dict[max_zone_id]

print("Q5: Pickup zone with largest total_amount on Nov 18:", max_zone_name)


Q5: Pickup zone with largest total_amount on Nov 18: East Harlem North


In [24]:
east_harlem_north_id = df_taxi[df_taxi["Zone"] == "East Harlem North"]["LocationID"].values[0]

# Filter trips with that pickup zone in November 2025
df_ehn = df_trip[
    (df_trip["PULocationID"] == east_harlem_north_id) &
    (df_trip["lpep_pickup_datetime"] >= "2025-11-01") &
    (df_trip["lpep_pickup_datetime"] < "2025-12-01")
]

# Sum tips by dropoff location
tip_total = df_ehn.groupby("DOLocationID")["tip_amount"].max()
max_tip_id = tip_total.idxmax()
max_tip_zone = zones_dict[max_tip_id]

print("Q6: Dropoff zone with largest tip for East Harlem North pickups:", max_tip_zone)

Q6: Dropoff zone with largest tip for East Harlem North pickups: Yorkville West


In [23]:
tip_total.idxmax()

np.int32(236)